# RAG from Scratch to Production — Episode 17/20
## FastAPI, streaming and the background worker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/17_fastapi_streaming_worker.ipynb)

Sixteen episodes built the RAG pipeline. Now it has to become an **app**: an API that many people use at once, answers that appear while you wait, and uploads that don't time out. The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) does this with **FastAPI**, a **streaming** endpoint and a **background worker**.

In this episode we run a small FastAPI server **inside the notebook**, with the same shape as production, and measure three things users actually feel:
1. While one person's question is being answered, can anyone else use the app?
2. How long until the **first word** of an answer appears?
3. How long after an upload is the document searchable?

**Needs an LLM:** local Ollama or Gemini (free key). About 15 calls.

## 0. Setup

In [ ]:
%pip install -q fastapi uvicorn httpx requests google-generativeai sentence-transformers

In [1]:
import os, json, time, asyncio, threading, statistics, requests
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)


def generate(prompt: str) -> str:
    """A normal, BLOCKING call, exactly like production's GeminiService / OllamaService."""
    if PROVIDER == "gemini":
        return _model.generate_content(prompt, generation_config={"temperature": 0}).text
    return requests.post("http://localhost:11434/api/chat", json={
        "model": OLLAMA_MODEL, "messages": [{"role": "user", "content": prompt}], "stream": False,
        "options": {"temperature": 0}}, timeout=300).json()["message"]["content"]


def stream_tokens(prompt: str):
    """The same answer, but token by token as the model writes it."""
    if PROVIDER == "gemini":
        for chunk in _model.generate_content(prompt, generation_config={"temperature": 0}, stream=True):
            if chunk.text:
                yield chunk.text
        return
    with requests.post("http://localhost:11434/api/chat", json={
            "model": OLLAMA_MODEL, "messages": [{"role": "user", "content": prompt}], "stream": True,
            "options": {"temperature": 0}}, stream=True, timeout=300) as r:
        for line in r.iter_lines():
            if line:
                token = json.loads(line).get("message", {}).get("content", "")
                if token:
                    yield token


generate("Say OK")   # warm-up
print(f"Using {PROVIDER}")

Using ollama


A small RAG step to serve: answer from one handbook section, then a second call that checks the answer, like production's hallucination guard (Episode 15). So a chat costs **2 LLM calls**, as in production.

In [2]:
CONTEXT = ("Remote Work: Employees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day "
           "for all teams. Remote days must be logged in the HR portal by Friday of the previous week.")
QUESTION = "Explain the remote work policy in two sentences."
PROMPT = f"Answer only from the context.\nContext: {CONTEXT}\nQuestion: {QUESTION}\nAnswer:"

def rag_answer() -> str:
    answer = generate(PROMPT)
    generate(f"Is this answer supported by the context? Context: {CONTEXT} Answer: {answer}")   # the guard
    return answer

## 1. The app

The same endpoints as production, plus fixed versions to compare:

| Endpoint | Like production? | What it does |
|---|---|---|
| `GET /health` | ✅ | Instant "I'm alive" |
| `POST /chat` | ✅ | `async def`, with blocking LLM calls inside |
| `POST /chat-threaded` | fix | Same work, moved to a thread |
| `POST /chat/stream-reveal` | ✅ | Finish everything, then reveal the answer word by word (production's `/chat/stream`) |
| `POST /chat/stream-real` | alternative | Stream tokens as the model writes them |

In [3]:
app = FastAPI()

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/chat")
async def chat():
    # Production's pattern: an async endpoint that calls blocking code directly.
    return {"answer": rag_answer()}

@app.post("/chat-threaded")
async def chat_threaded():
    # The fix: run the blocking work in a thread, so the server stays free.
    return {"answer": await asyncio.to_thread(rag_answer)}

@app.post("/chat/stream-reveal")
async def chat_stream_reveal():
    # Production's /chat/stream: the whole pipeline finishes first, then words are revealed.
    answer = await asyncio.to_thread(rag_answer)
    async def events():
        for word in answer.split(" "):
            yield f"data: {json.dumps(word + ' ')}\n\n"
            await asyncio.sleep(0.02)          # production's reveal delay per word
    return StreamingResponse(events(), media_type="text/event-stream")

@app.post("/chat/stream-real")
def chat_stream_real():
    # Real streaming: forward each token as the model produces it.
    # (A plain generator: FastAPI iterates it in a thread, so it doesn't block either.)
    def events():
        for token in stream_tokens(PROMPT):
            yield f"data: {json.dumps(token)}\n\n"
    return StreamingResponse(events(), media_type="text/event-stream")


server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8765, log_level="error"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
BASE = "http://127.0.0.1:8765"
print(requests.get(f"{BASE}/health").json())

{'status': 'ok'}


## 2. Experiment 1: can anyone else use the app?

Send one chat request, and while it runs, keep pinging `/health` every 0.1 seconds, like a second user, or a load balancer checking the server is alive.

In [4]:
def health_during(path):
    pings, finished = [], threading.Event()
    def one_chat():
        requests.post(f"{BASE}{path}")
        finished.set()
    start = time.perf_counter()
    threading.Thread(target=one_chat).start()
    time.sleep(0.2)
    while not finished.is_set():
        t = time.perf_counter()
        requests.get(f"{BASE}/health")
        pings.append(time.perf_counter() - t)
        time.sleep(0.1)
    took = time.perf_counter() - start
    print(f"{path:<16} chat took {took:.1f}s   /health answered {len(pings):>2} times, slowest {max(pings) * 1000:,.0f} ms")

health_during("/chat")
health_during("/chat-threaded")

/chat            chat took 3.2s   /health answered  1 times, slowest 2,933 ms


/chat-threaded   chat took 2.7s   /health answered 23 times, slowest 5 ms


With production's pattern, `/health` gets answered **once** during the whole chat, and that one answer waits seconds, until the chat finishes. Nobody else gets anything either.

**Why:** an `async def` endpoint runs on FastAPI's **event loop**, a single thread that juggles every request. `async` code hands control back while it waits (`await`). A normal blocking call (an HTTP request to the LLM, an embedding, a reranker) **doesn't**, so the whole loop freezes until it returns.

The fix is one line: run blocking work with `await asyncio.to_thread(...)` (or declare the endpoint with plain `def`, which FastAPI runs in a thread pool). The chat takes as long, but everyone else is served in milliseconds.

## 3. Experiment 2: time to the first word

In [5]:
def first_word(path):
    start, first = time.perf_counter(), None
    with requests.post(f"{BASE}{path}", stream=True) as r:
        for line in r.iter_lines():
            if line and first is None:
                first = time.perf_counter() - start
    print(f"{path:<20} first word after {first:5.2f}s   complete after {time.perf_counter() - start:5.2f}s")

first_word("/chat/stream-reveal")
first_word("/chat/stream-real")

/chat/stream-reveal  first word after  2.68s   complete after  3.42s


/chat/stream-real    first word after  0.05s   complete after  0.93s


Production's `/chat/stream` makes the user wait for the **entire** pipeline (answer + guard) before showing the first word, then spends extra time on the 20 ms-per-word reveal. Total time is **longer** than without "streaming".

Real token streaming shows the first word almost immediately. So why doesn't production do it? Because of the **hallucination guard**: with real streaming, the user is already reading the answer before it has been checked. The production code says this choice was deliberate.

There's no free lunch, but there are middle grounds:
- **Stream, then check**: show tokens live, run the guard after, and attach a warning (or replace the answer) if it fails.
- **Stream only when it's safe**: skip the guard on refusals (Episode 15), stream when the reranker is confident (Episode 11's bands), and fall back to reveal otherwise.
- **At least drop the artificial delay**: the reveal makes the full answer arrive later, not sooner.

## 4. Experiment 3: the background worker

Parsing, chunking and embedding a big PDF can take minutes, too long for an HTTP request. So production's `/upload` saves the file, marks it **pending**, and returns immediately. A separate **worker** process checks the database **every 5 seconds** for pending documents and processes them.

Let's measure what the polling costs, compared with a worker that's **notified** the moment a document arrives. Each "upload" arrives at a random time, and processing (embedding a few chunks) is real.

In [6]:
import random
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
CHUNKS = [CONTEXT] * 20


def process(document):
    embedder.encode(CHUNKS)              # the real work: chunk + embed
    document["status"] = "completed"
    document["done_at"] = time.perf_counter()


def polling_worker(db, stop, interval=5.0):
    """Production's worker: look for pending documents every few seconds."""
    while not stop.is_set():
        for doc in db:
            if doc["status"] == "pending":
                process(doc)
        stop.wait(interval)


def notified_worker(queue, stop):
    """Alternative: wake up as soon as an upload is queued."""
    while not stop.is_set():
        try:
            process(queue.get(timeout=0.5))
        except Exception:
            pass


def run(worker_kind, uploads=5):
    import queue as q
    db, inbox, stop = [], q.Queue(), threading.Event()
    target = (lambda: polling_worker(db, stop)) if worker_kind == "polling" else (lambda: notified_worker(inbox, stop))
    threading.Thread(target=target, daemon=True).start()
    random.seed(1)
    for _ in range(uploads):
        time.sleep(random.uniform(0.5, 4.0))                     # users upload at random moments
        doc = {"status": "pending", "uploaded_at": time.perf_counter()}
        db.append(doc)
        inbox.put(doc)
    while any(d["status"] != "completed" for d in db):
        time.sleep(0.05)
    stop.set()
    waits = [d["done_at"] - d["uploaded_at"] for d in db]
    print(f"{worker_kind:<9} upload → searchable: average {statistics.mean(waits):.1f}s, worst {max(waits):.1f}s")


run("polling")
run("notified")

polling   upload → searchable: average 2.7s, worst 4.3s


notified  upload → searchable: average 0.0s, worst 0.1s


The processing itself is quick here. Most of the polling worker's delay is just **waiting for the next check**: on average about half the interval, up to the full 5 seconds.

Polling isn't wrong, though. Production's worker uses the **database** as its queue, which has real advantages: a pending document survives a crash or a redeploy, the web app and the worker share nothing but the database, and there's no extra queue service to run. A notified worker with an in-memory queue loses everything on restart.

Middle grounds if the delay matters: a shorter interval, or a database notification (PostgreSQL's `LISTEN/NOTIFY`) that wakes the worker while keeping the database as the source of truth.

## 5. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot):
- Runs **one** uvicorn process (`exec uvicorn app.main:app`, no `--workers`), and its chat endpoints are `async def` with blocking LLM, embedding and reranker calls inside. So while one chat runs, **every** other request waits, including the Docker health check (10 s timeout, 3 retries). Long local-model chats could make the container look unhealthy.
- Streams by **revealing** a finished answer, so the guard always runs first. Safe, but the first word appears after the whole pipeline, and the reveal adds time.
- Uses a **separate worker process** that polls PostgreSQL every 5 seconds. Durable and simple, at the cost of a few seconds' wait after each upload. It also keeps the heavy ML libraries out of the web process, a good call.

## Recap

| What users feel | Production's design | What we measured | Fix |
|---|---|---|---|
| "The app froze while someone else asked something" | `async def` + blocking calls | `/health` waited seconds during one chat | `await asyncio.to_thread(...)` |
| "How long until I see something?" | Reveal after the full pipeline | First word only after answer + guard | Real streaming, with a check after, or when safe |
| "My upload says pending" | Poll every 5 s | Extra wait ≈ half the interval | Shorter interval or `LISTEN/NOTIFY` |

**The rule:** in an async web server, anything slow must either be `await`ed or moved to a thread. And measure what the **user** waits for, not just how long the work takes.

In [7]:
server.should_exit = True   # stop the notebook's server

**Next, Episode 18: auth, rate limiting and access control**: who can ask, how often, and which documents they can see.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)